In [83]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import accuracy_score

In [47]:
df = pd.read_csv('train.csv')

In [48]:
df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace=True)

In [49]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [50]:
# Step 1 -> train/test/split
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [51]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S
...,...,...,...,...,...,...,...
106,3,female,21.0,0,0,7.6500,S
270,1,male,NaN,0,0,31.0000,S
860,3,male,41.0,2,0,14.1083,S
435,1,female,14.0,1,2,120.0000,S


In [52]:
X_train.isnull().sum()

Pclass        0
Sex           0
Age         140
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [53]:
X_test.isnull().sum()

Pclass       0
Sex          0
Age         37
SibSp        0
Parch        0
Fare         0
Embarked     0
dtype: int64

In [ ]:
# Imputation transformer
trf1 = ColumnTransformer(
 [
    ('mean_imputer', SimpleImputer(strategy='mean'), [2]),
    ('frequent_imputer', SimpleImputer(strategy="most_frequent"),[6] )
 ], remainder="passthrough"
)

In [ ]:
# One hot encoding
trf2 = ColumnTransformer([
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.int16), [1, 3])
], remainder='passthrough')

In [ ]:
# Scaling
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0, 10))
], remainder='passthrough')

In [ ]:
# Feature Selection
trf4 = SelectKBest(score_func=chi2, k=8)

In [ ]:
# Train the model
trf5 = DecisionTreeClassifier()

### Create Pipeline

In [59]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4),
    ('trf5', trf5)
])

## Pipeline Vs make_pipeline
Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)


In [33]:
pipe = make_pipeline(trf1, trf2, trf3, trf4, trf5)

In [60]:
pipe.fit(X_train, y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('mean_imputer', ...), ('frequent_imputer', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [91]:
# After encoding (trf2)
X_after_trf2 = pipe[:-2].transform(X_train)
X_after_trf2
# print("After encoding:", X_after_trf2.shape)

array([[0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.0556283 ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.02537431],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.01546857],
       ...,
       [0.        , 0.        , 1.        , ..., 0.25      , 0.        ,
        0.02753757],
       [0.        , 0.        , 1.        , ..., 0.125     , 0.33333333,
        0.2342244 ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.16666667,
        0.15085515]], shape=(712, 10))

In [94]:
# After all transformations (before the classifier)
X_transformed = pipe[:-1].transform(X_train)

# Get feature names from the pipeline
feature_names = pipe[:-1].get_feature_names_out()

# Create DataFrame with proper column names
df_transformed = pd.DataFrame(X_transformed, columns=feature_names)
# print(df_transformed)
df_transformed.head()
# print(df_transformed.info())

,scale__ohe__frequent_imputer__Embarked_C,scale__ohe__frequent_imputer__Embarked_S,scale__ohe__remainder__Sex_female,scale__ohe__remainder__Sex_male,scale__remainder__remainder__Pclass,scale__remainder__remainder__SibSp,scale__remainder__remainder__Parch,scale__remainder__remainder__Fare
0,0.0,1.0,0.0,1.0,0.0,0.000,0.000000,0.055628
1,0.0,1.0,0.0,1.0,0.5,0.000,0.000000,0.025374
2,0.0,1.0,0.0,1.0,1.0,0.000,0.000000,0.015469
3,0.0,1.0,0.0,1.0,1.0,0.125,0.000000,0.015330
4,0.0,1.0,1.0,0.0,1.0,0.500,0.333333,0.061045


In [89]:
pd.DataFrame(pipe[:-1].transform(X_train)) 

,0,1,2,3,4,5,6,7
0,0.0,1.0,0.0,1.0,0.0,0.000,0.000000,0.055628
1,0.0,1.0,0.0,1.0,0.5,0.000,0.000000,0.025374
2,0.0,1.0,0.0,1.0,1.0,0.000,0.000000,0.015469
3,0.0,1.0,0.0,1.0,1.0,0.125,0.000000,0.015330
4,0.0,1.0,1.0,0.0,1.0,0.500,0.333333,0.061045
...,...,...,...,...,...,...,...,...
707,0.0,1.0,1.0,0.0,1.0,0.000,0.000000,0.014932
708,0.0,1.0,0.0,1.0,0.0,0.000,0.000000,0.060508
709,0.0,1.0,0.0,1.0,1.0,0.250,0.000000,0.027538
710,0.0,1.0,1.0,0.0,0.0,0.125,0.333333,0.234224


In [61]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('mean_imputer', SimpleImputer(), [2]),
                                 ('frequent_imputer',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe',
                                  OneHotEncoder(dtype=<class 'numpy.int16'>,
                                                handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 3])]),
 'trf3': ColumnTransformer(remainder='passthrough',
                   transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x7bb952baa480>),
 'trf5': DecisionTreeClassifier()}

In [75]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [68]:
from sklearn import set_config
set_config(display='diagram')

In [76]:
y_pred = pipe.predict(X_test)

In [77]:
y_pred

array([0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1])

In [80]:
accuracy_score(y_test, y_pred)

0.7877094972067039

### Cross Validation using Pipeline


In [ ]:
# cross validation using cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.7838569880823403)

### GridSearch using Pipeline

In [82]:
# gridsearchcv
params = {
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [84]:
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'trf5__max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('mean_imputer', ...), ('frequent_imputer', ...)]"


In [85]:
grid.best_score_

np.float64(0.8033093666896484)

In [86]:
grid.best_params_

{'trf5__max_depth': 3}

## Exporting the Pipeline

In [88]:
import pickle
pickle.dump(pipe, open('pipe.pkl', 'wb'))